# InterviewIQ — Answer Quality Model

This notebook trains an explainable classifier using the labelled InterviewIQ dataset. It predicts whether an answer is **weak**, **average**, or **strong**.

## 1. Import libraries

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

## 2. Load the dataset
The CSV file is included in the project at `data/training_answers.csv`.

In [ ]:
DATASET_PATH = Path('data/training_answers.csv')
data = pd.read_csv(DATASET_PATH)
print(f'Dataset rows: {len(data)}')
data.head()

In [ ]:
# The labels are balanced for this starter demonstration dataset.
data['label'].value_counts()

## 3. Create the train/test split
We keep 30% of the answers unseen for testing. `random_state=42` gives the same split every run.

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    data['answer'], data['label'],
    test_size=0.30, random_state=42, stratify=data['label']
)
print(f'Training rows: {len(x_train)} | Test rows: {len(x_test)}')

## 4. Build and train the model
TF-IDF captures meaningful words and phrases. Word count is an additional transparent feature because detailed answers tend to score higher in this starter dataset. Logistic Regression is lightweight and beginner-friendly.

In [ ]:
def answer_length(texts):
    return np.array([[len(str(text).split())] for text in texts])

model = Pipeline([
    ('features', FeatureUnion([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), stop_words='english')),
        ('answer_length', Pipeline([
            ('count_words', FunctionTransformer(answer_length, validate=False)),
            ('scale', StandardScaler()),
        ])),
    ])),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)),
])

model.fit(x_train, y_train)
print('Training complete.')

## 5. Evaluate accuracy

In [ ]:
predictions = model.predict(x_test)
accuracy = accuracy_score(y_test, predictions)
print(f'Test accuracy: {accuracy * 100:.2f}%')
print('\nClassification report:')
print(classification_report(y_test, predictions, zero_division=0))
print('Confusion matrix (rows = actual, columns = predicted):')
print(confusion_matrix(y_test, predictions, labels=['weak', 'average', 'strong']))

## 6. Predict a new answer
Change the text below and run this cell to test your own answer.

In [ ]:
new_answer = '''An API is a contract that allows a browser and backend service to communicate. The client sends an HTTP request to an endpoint and receives a structured response, often JSON.'''
prediction = model.predict([new_answer])[0]
probabilities = dict(zip(model.classes_, model.predict_proba([new_answer])[0].round(3)))
print('Predicted quality:', prediction)
print('Class probabilities:', probabilities)

## Viva note
This is a **small curated starter dataset**, so its score is for demonstrating the complete machine-learning workflow. For a real research result, collect many more real answers and label them manually before calculating final accuracy.